# 03 — Music Matching (1 photo = 1 musique + playlist souvenir)

Associe chaque **photo** ET chaque **groupe** à la musique la plus cohérente.

| Priorité | Méthode |
|---|---|
| 1 | Override manuel (`config.yaml`) |
| 2 | Matching temporel Spotify (stream dans la fenêtre de la photo ±60min) |
| 3 | Matching sémantique (embedding caption ↔ moment tags) |

**Sorties** :
- `music_matches/` — 1 musique par groupe
- `photo_music/` — 1 musique par photo
- `group_playlists/` — tous les streams dans la fenêtre du groupe, tri chronologique

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)

import json
import yaml
import numpy as np
import pandas as pd

from config import (
    build_spark_session,
    MEMORY_ALBUM_DIR,
    WAREHOUSE,
    MUSIC_LIBRARY_PATH,
    MODELS_CACHE_DIR,
)

CENTROIDS_DIR    = os.path.join(MEMORY_ALBUM_DIR, 'scene_centroids')
SCENES_DIR       = os.path.join(MEMORY_ALBUM_DIR, 'scenes')
STREAMS_DIR      = os.path.join(WAREHOUSE, 'spotify_streams')
MATCHES_DIR      = os.path.join(MEMORY_ALBUM_DIR, 'music_matches')
PHOTO_MUSIC_DIR  = os.path.join(MEMORY_ALBUM_DIR, 'photo_music')
PLAYLISTS_DIR    = os.path.join(MEMORY_ALBUM_DIR, 'group_playlists')

print(f"Centroids : {CENTROIDS_DIR}")
print(f"Output    : {MATCHES_DIR}")
print(f"          : {PHOTO_MUSIC_DIR}")
print(f"          : {PLAYLISTS_DIR}")

Centroids : /opt/spark/data/warehouse/memory_album/scene_centroids
Output    : /opt/spark/data/warehouse/memory_album/music_matches
          : /opt/spark/data/warehouse/memory_album/photo_music
          : /opt/spark/data/warehouse/memory_album/group_playlists


In [2]:
# ── 1. PARAMÈTRES ─────────────────────────────────────────────────────────────
_cfg_path = os.path.join(_d, 'config.yaml')
with open(_cfg_path, encoding='utf-8') as _f:
    _cfg = yaml.safe_load(_f)

_ma = _cfg.get('memory_album', {})

TEMPORAL_WINDOW_MIN = int(_ma.get('temporal_window_minutes', 60))
MANUAL_OVERRIDES    = _ma.get('manual_overrides', [])

print(f"Fenêtre temporelle : ±{TEMPORAL_WINDOW_MIN} min")
print(f"Overrides manuels  : {len(MANUAL_OVERRIDES)} entrée(s)")

Fenêtre temporelle : ±60 min
Overrides manuels  : 5 entrée(s)


In [3]:
# ── 2. SESSION SPARK ──────────────────────────────────────────────────────────
from pyspark.sql import functions as F

spark = build_spark_session('MyDigitalTwin-MemoryAlbum-MusicMatching')
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Spark 3.5.5


26/05/12 16:59:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
# ── 3. CHARGEMENT DES DONNÉES ─────────────────────────────────────────────────
df_centroids = spark.read.format('delta').load(CENTROIDS_DIR)
print(f"Groupes : {df_centroids.count()}")

df_scenes_all = spark.read.format('delta').load(SCENES_DIR)
scenes_pdf    = df_scenes_all.select(
    'photo_id', 'scene_id', 'exif_date', 'path', 'caption'
).toPandas()
scenes_pdf['exif_date'] = pd.to_datetime(scenes_pdf['exif_date'], errors='coerce')
print(f"Photos  : {len(scenes_pdf)}")

_has_streams = os.path.exists(STREAMS_DIR)
if _has_streams:
    df_streams = spark.read.parquet(STREAMS_DIR)
    print(f"Streams : {df_streams.count()}")
    print(f"Colonnes streams : {df_streams.columns}")
else:
    df_streams = None
    print("⚠ Pas de dossier spotify_streams")

_has_library = os.path.exists(MUSIC_LIBRARY_PATH)
if _has_library:
    with open(MUSIC_LIBRARY_PATH, encoding='utf-8') as _f:
        music_library = json.load(_f)
    print(f"Bibliothèque : {len(music_library)} titres")
    n_with_moments = sum(1 for t in music_library if t.get('moments'))
    print(f"  dont {n_with_moments} avec moment tags")
else:
    music_library = []
    print("⚠ music_library.json introuvable")

26/05/12 16:59:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Groupes : 26
Photos  : 64
Streams : 123033
Colonnes streams : ['artistName', 'trackName', 'msPlayed', 'minutes_played', 'trackUri', 'skipped', 'shuffle', 'listen_ts', 'listen_year', 'listen_month', 'listen_hour', 'listen_weekday', 'listen_week', 'is_night']
Bibliothèque : 75 titres
  dont 75 avec moment tags


In [5]:
# ── 4. PRÉPARATION STREAMS ────────────────────────────────────────────────────
# Normalise les noms de colonnes Spotify (varient selon la version d'export)

streams_pdf = pd.DataFrame()

if _has_streams and df_streams is not None:
    stream_cols = df_streams.columns
    ts_col      = next((c for c in ['stream_ts', 'played_at', 'ts', 'listen_ts'] if c in stream_cols), None)
    track_id_col    = next((c for c in ['track_id', 'trackId', 'spotify_track_uri', 'trackUri'] if c in stream_cols), None)
    track_name_col  = next((c for c in ['track_name', 'trackName', 'master_metadata_track_name'] if c in stream_cols), None)
    artist_col      = next((c for c in ['artist_name', 'artistName', 'master_metadata_album_artist_name'] if c in stream_cols), None)

    missing = [n for n, c in [('timestamp', ts_col), ('track_id', track_id_col),
                               ('track_name', track_name_col), ('artist', artist_col)] if c is None]
    if missing:
        print(f"⚠ Colonnes manquantes dans streams : {missing} — matching temporel désactivé")
    else:
        streams_pdf = (
            df_streams
            .select(
                F.col(ts_col).cast('timestamp').alias('stream_ts'),
                F.col(track_id_col).alias('track_id'),
                F.col(track_name_col).alias('track_name'),
                F.col(artist_col).alias('artist_name'),
            )
            .filter(F.col('track_id').isNotNull())
            .toPandas()
        )
        streams_pdf['stream_ts'] = pd.to_datetime(streams_pdf['stream_ts'], errors='coerce')
        streams_pdf = streams_pdf.dropna(subset=['stream_ts'])
        print(f"Streams utilisables : {len(streams_pdf)}")

Streams utilisables : 122292


In [6]:
# ── 5. PRÉPARATION SÉMANTIQUE ─────────────────────────────────────────────────
# Encode les moment tags de chaque titre avec sentence-transformers MiniLM

semantic_ready = False
moment_embs    = None
library_with_moments = [t for t in music_library if t.get('moments')]

if library_with_moments:
    try:
        from sentence_transformers import SentenceTransformer
        model_sem = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=MODELS_CACHE_DIR)
        moment_texts = [
            ', '.join(t['moments']) if isinstance(t['moments'], list) else str(t['moments'])
            for t in library_with_moments
        ]
        moment_embs = model_sem.encode(moment_texts, normalize_embeddings=True)
        semantic_ready = True
        print(f"Modèle sémantique chargé — {len(library_with_moments)} titres avec moment tags")
    except Exception as e:
        print(f"⚠ Sémantique indisponible : {e}")
else:
    print("⚠ Aucun moment tag dans la bibliothèque — matching sémantique désactivé")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modèle sémantique chargé — 75 titres avec moment tags


In [7]:
# ── 6. FONCTIONS DE MATCHING ──────────────────────────────────────────────────
W_SEC = TEMPORAL_WINDOW_MIN * 60

# Normalise track_id : accepte "spotify:track:XXX" ou juste "XXX"
def _normalize_tid(tid: str) -> str:
    if not tid:
        return tid
    return tid if tid.startswith('spotify:track:') else f'spotify:track:{tid}'

lib_lookup = {t['track_id']: t for t in music_library}

# Override manuel : photo_id → track_id (normalisé)
photo_overrides = {
    ov['photo_id']: _normalize_tid(ov['track_id'])
    for ov in MANUAL_OVERRIDES
    if ov.get('photo_id') and ov.get('track_id')
}

# Fallback global : titre le plus joué dans l'historique Spotify
_global_fallback = None
if not streams_pdf.empty:
    _top = streams_pdf.groupby(['track_id', 'track_name', 'artist_name']).size().idxmax()
    _tid, _tname, _tartist = _top
    _meta = lib_lookup.get(_normalize_tid(_tid), {})
    _global_fallback = {
        'track_id'       : _normalize_tid(_tid),
        'track_name'     : _tname,
        'artist_name'    : _tartist,
        'album_cover_url': _meta.get('album_cover_url', ''),
        'match_type'     : 'fallback',
        'match_score'    : 0.0,
    }
elif music_library:
    t = music_library[0]
    _global_fallback = {
        'track_id': t['track_id'], 'track_name': t['track_name'],
        'artist_name': t.get('artist', ''), 'album_cover_url': t.get('album_cover_url', ''),
        'match_type': 'fallback', 'match_score': 0.0,
    }
print(f"Fallback global : {_global_fallback['track_name'] if _global_fallback else 'aucun'}")

# Contexte temporel : génère une phrase de mood basée sur l'heure
# Utilisé pour le matching sémantique quand le caption est vide (ex: OpenCLIP sans texte)
def _time_context(ts) -> str:
    if ts is None or pd.isna(ts):
        return ''
    dt = pd.Timestamp(ts)
    h  = dt.hour
    m  = dt.month
    # Saison
    season = 'winter' if m in (12, 1, 2) else 'spring' if m in (3, 4, 5) else 'summer' if m in (6, 7, 8) else 'autumn'
    # Créneau
    if   0 <= h < 6:  slot = 'late night dark quiet'
    elif 6 <= h < 12: slot = 'morning fresh start energy'
    elif 12 <= h < 18: slot = 'afternoon chill sunny'
    elif 18 <= h < 23: slot = 'evening friends social vibes'
    else:             slot = 'night out party'
    return f'{slot} {season}'


def temporal_match(ts, window_sec: int = None) -> dict | None:
    """Trouve le stream Spotify le plus joué dans la fenêtre ±window_sec autour de ts."""
    if streams_pdf.empty or ts is None or pd.isna(ts):
        return None
    ws  = window_sec or W_SEC
    ts  = pd.Timestamp(ts)
    mask = (
        (streams_pdf['stream_ts'] >= ts - pd.Timedelta(seconds=ws)) &
        (streams_pdf['stream_ts'] <= ts + pd.Timedelta(seconds=ws))
    )
    hits = streams_pdf[mask]
    if hits.empty:
        return None
    best = hits.groupby(['track_id', 'track_name', 'artist_name']).size().idxmax()
    tid, tname, artist = best
    meta = lib_lookup.get(_normalize_tid(tid), {})
    return {
        'track_id'       : _normalize_tid(tid),
        'track_name'     : tname,
        'artist_name'    : artist,
        'album_cover_url': meta.get('album_cover_url', ''),
        'match_type'     : 'temporal',
        'match_score'    : float(hits[hits['track_id'] == tid].shape[0]),
    }


def semantic_match(text: str) -> dict | None:
    """Trouve le titre dont les moment tags sont les plus proches du texte."""
    if not semantic_ready or not text or not text.strip():
        return None
    emb    = model_sem.encode([text], normalize_embeddings=True)[0]
    scores = moment_embs @ emb
    best_i = int(np.argmax(scores))
    score  = float(scores[best_i])
    if score < 0.15:   # seuil minimal de pertinence
        return None
    best_t = library_with_moments[best_i]
    return {
        'track_id'       : best_t['track_id'],
        'track_name'     : best_t['track_name'],
        'artist_name'    : best_t.get('artist', ''),
        'album_cover_url': best_t.get('album_cover_url', ''),
        'match_type'     : 'semantic',
        'match_score'    : score,
    }


def best_match(photo_id: str, ts, caption: str) -> dict:
    """
    Priorité : manuel → temporel (±60min) → sémantique caption → sémantique contexte horaire
              → temporel élargi (±3h) → fallback global
    """
    # 1. Override manuel
    if photo_id in photo_overrides:
        tid  = photo_overrides[photo_id]
        meta = lib_lookup.get(tid, {})
        return {
            'track_id'       : tid,
            'track_name'     : meta.get('track_name', tid.replace('spotify:track:', '')),
            'artist_name'    : meta.get('artist', ''),
            'album_cover_url': meta.get('album_cover_url', ''),
            'match_type'     : 'manual',
            'match_score'    : 1.0,
        }
    # 2. Temporel ±60min
    m = temporal_match(ts)
    if m: return m
    # 3. Sémantique sur le caption (si disponible)
    m = semantic_match(caption)
    if m: return m
    # 4. Sémantique sur le contexte horaire (fallback sémantique)
    m = semantic_match(_time_context(ts))
    if m: return m
    # 5. Temporel élargi ±3h
    m = temporal_match(ts, window_sec=3 * 3600)
    if m: return m
    # 6. Fallback global (titre le plus joué)
    return _global_fallback or {
        'track_id': None, 'track_name': None, 'artist_name': None,
        'album_cover_url': '', 'match_type': 'none', 'match_score': 0.0
    }

print("Fonctions de matching prêtes")

Fallback global : Applaudissement
Fonctions de matching prêtes


In [8]:
# ── 7. MATCHING PAR PHOTO ─────────────────────────────────────────────────────
photo_results = []

for _, row in scenes_pdf.iterrows():
    m = best_match(
        photo_id = row['photo_id'],
        ts       = row['exif_date'],
        caption  = row.get('caption', '') or '',
    )
    photo_results.append({
        'photo_id'       : row['photo_id'],
        'scene_id'       : int(row['scene_id']),
        'path'           : row['path'],
        'exif_date'      : row['exif_date'],
        'track_id'       : m['track_id'],
        'track_name'     : m['track_name'],
        'artist_name'    : m['artist_name'],
        'album_cover_url': m['album_cover_url'],
        'match_type'     : m['match_type'],
        'match_score'    : m['match_score'],
    })

photo_pdf = pd.DataFrame(photo_results)
print(f"Matching photo terminé : {len(photo_pdf)} photos")
print(photo_pdf['match_type'].value_counts().to_string())

Matching photo terminé : 64 photos
match_type
semantic    29
temporal    16
fallback    14
manual       5


In [9]:
# ── 8. MATCHING PAR GROUPE ────────────────────────────────────────────────────
# Pour le groupe : on prend la musique la plus fréquente parmi les photos du groupe
# (si temporel domine), sinon on applique le même pipeline sur le centroïde.

centroids_pdf = df_centroids.toPandas()
group_results = []

for _, row in centroids_pdf.iterrows():
    sid = int(row['scene_id'])

    # Votes des photos du groupe (hors fallback et none)
    group_photos = photo_pdf[photo_pdf['scene_id'] == sid]
    strong = group_photos[group_photos['match_type'].isin(['manual', 'temporal', 'semantic'])]

    if not strong.empty:
        # Musique la plus fréquente dans les matchs forts
        best_tid = strong['track_id'].value_counts().idxmax()
        best_row = strong[strong['track_id'] == best_tid].iloc[0]
        m = {
            'track_id'       : best_row['track_id'],
            'track_name'     : best_row['track_name'],
            'artist_name'    : best_row['artist_name'],
            'album_cover_url': best_row['album_cover_url'],
            'match_type'     : best_row['match_type'],
            'match_score'    : float(strong['track_id'].eq(best_tid).sum()),
        }
    else:
        # Fallback sur le centroïde
        caption = str(row.get('representative_caption', '') or '')
        ts      = row.get('timestamp_start')
        m = best_match(photo_id='__group__', ts=ts, caption=caption)

    group_results.append({
        'scene_id'       : sid,
        'scene_name'     : row['scene_name'],
        'photo_count'    : int(row['photo_count']),
        'track_id'       : m['track_id'],
        'track_name'     : m['track_name'],
        'artist_name'    : m['artist_name'],
        'album_cover_url': m['album_cover_url'],
        'match_type'     : m['match_type'],
        'match_score'    : m['match_score'],
        'timestamp_start': row.get('timestamp_start'),
        'timestamp_end'  : row.get('timestamp_end'),
    })

group_pdf = pd.DataFrame(group_results)
print("\n── Musique par groupe ────────────────────────────")
print(group_pdf[['scene_id', 'scene_name', 'track_name', 'artist_name', 'match_type']].to_string(index=False))


── Musique par groupe ────────────────────────────
 scene_id                scene_name                              track_name        artist_name match_type
        0      Matin · 22 juin 2022                         Applaudissement            Natoxie   temporal
        1      Soirée · 21 jan 2023                               Pineapple               Leto   semantic
        2  Après-midi · 14 fév 2023                                Fentanyl    Freeze corleone   temporal
        3        Nuit · 27 mai 2023                   My Love Mine All Mine             Mitski     manual
        4     Soirée · 17 juil 2023                        Tití Me Preguntó          Bad Bunny   semantic
        5      Matin · 16 août 2023                               Feel Good   Charlotte Cardin   semantic
        6        Nuit · 14 sep 2023           All Eyez On Me (ft. Big Syke)               2Pac   temporal
        7  Après-midi · 16 déc 2023                               Pineapple               Leto   sem

In [10]:
# ── 9. PLAYLIST SOUVENIR PAR GROUPE ──────────────────────────────────────────
# Tous les streams Spotify dans la fenêtre temporelle du groupe, tri chronologique

playlist_rows = []

for _, row in centroids_pdf.iterrows():
    sid  = int(row['scene_id'])
    ts0  = row.get('timestamp_start')
    ts1  = row.get('timestamp_end')

    if streams_pdf.empty or ts0 is None or pd.isna(ts0):
        continue

    ts0 = pd.Timestamp(ts0)
    ts1 = pd.Timestamp(ts1) if ts1 and not pd.isna(ts1) else ts0

    window_start = ts0 - pd.Timedelta(seconds=W_SEC)
    window_end   = ts1 + pd.Timedelta(seconds=W_SEC)

    hits = streams_pdf[
        (streams_pdf['stream_ts'] >= window_start) &
        (streams_pdf['stream_ts'] <= window_end)
    ].sort_values('stream_ts')

    for rank, (_, s) in enumerate(hits.iterrows()):
        meta = lib_lookup.get(s['track_id'], {})
        playlist_rows.append({
            'scene_id'       : sid,
            'scene_name'     : row['scene_name'],
            'rank'           : rank,
            'stream_ts'      : s['stream_ts'],
            'track_id'       : s['track_id'],
            'track_name'     : s['track_name'],
            'artist_name'    : s['artist_name'],
            'album_cover_url': meta.get('album_cover_url', ''),
        })

playlist_pdf = pd.DataFrame(playlist_rows)
print(f"Playlist souvenir : {len(playlist_pdf)} entrées pour {playlist_pdf['scene_id'].nunique() if not playlist_pdf.empty else 0} groupes")

Playlist souvenir : 165 entrées pour 12 groupes


In [11]:
# ── 10. ÉCRITURE DELTA ────────────────────────────────────────────────────────

# music_matches (groupe)
(
    spark.createDataFrame(group_pdf.astype({'scene_id': int, 'photo_count': int}))
    .write.format('delta').mode('overwrite').save(MATCHES_DIR)
)
print(f"✓ music_matches   → {MATCHES_DIR}")

# photo_music (photo individuelle)
(
    spark.createDataFrame(photo_pdf.astype({'scene_id': int}))
    .write.format('delta').mode('overwrite').save(PHOTO_MUSIC_DIR)
)
print(f"✓ photo_music     → {PHOTO_MUSIC_DIR}")

# group_playlists
if not playlist_pdf.empty:
    (
        spark.createDataFrame(playlist_pdf.astype({'scene_id': int, 'rank': int}))
        .write.format('delta').mode('overwrite').save(PLAYLISTS_DIR)
    )
    print(f"✓ group_playlists → {PLAYLISTS_DIR}")
else:
    print("⚠ Aucune playlist générée (pas de streams dans les fenêtres temporelles)")

print("\n── Récap final ────────────────────────────────────")
spark.read.format('delta').load(MATCHES_DIR)\
    .select('scene_id', 'scene_name', 'track_name', 'artist_name', 'match_type')\
    .orderBy('scene_id').show(truncate=50)

✓ music_matches   → /opt/spark/data/warehouse/memory_album/music_matches


✓ photo_music     → /opt/spark/data/warehouse/memory_album/photo_music
✓ group_playlists → /opt/spark/data/warehouse/memory_album/group_playlists

── Récap final ────────────────────────────────────
+--------+-------------------------+---------------------------------------+------------------+----------+
|scene_id|               scene_name|                             track_name|       artist_name|match_type|
+--------+-------------------------+---------------------------------------+------------------+----------+
|       0|     Matin · 22 juin 2022|                        Applaudissement|           Natoxie|  temporal|
|       1|     Soirée · 21 jan 2023|                              Pineapple|              Leto|  semantic|
|       2| Après-midi · 14 fév 2023|                               Fentanyl|   Freeze corleone|  temporal|
|       3|       Nuit · 27 mai 2023|                  My Love Mine All Mine|            Mitski|    manual|
|       4|    Soirée · 17 juil 2023|                

In [13]:
spark.stop()